# 03 - Machine Learning Model Training & Comparative Evaluation
## EV Charging Station Congestion & Grid Load Predictor

This notebook details:
1. Feature matrix and target separation strictly preventing data leakage.
2. Pipeline preprocessing using `ColumnTransformer` (OneHotEncoding + StandardScaler).
3. Training three multi-class models for Congestion Level:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
4. Training a Random Forest classifier for Grid Load Level proxy.
5. Comparative model evaluation across Accuracy, Precision, Recall, and F1-Score.
6. Confusion matrix analysis and model serialization via `joblib`.


### 1. Load Processed Dataset


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

df = pd.read_csv("../data/processed/ev_charging_project_2000.csv")
print(f"Loaded dataset: {df.shape}")
df.head(3)


### 2. Feature Selection & Data Leakage Prevention


In [ ]:
# Input features excluding identifiers (sessionId, userId) and targets (congestion_level, load_level, estimated_wait_min)
categorical_cols = ["station_id", "location_id", "day_of_week", "facility_type"]
numerical_cols = ["hour", "avg_energy_kwh", "avg_charge_time_min", "sessions_at_station_hour"]
feature_cols = categorical_cols + numerical_cols

X = df[feature_cols].copy()
y_congestion = df["congestion_level"].copy()
y_load = df["load_level"].copy()

print("Feature Matrix Shape:", X.shape)
print("Categorical features:", categorical_cols)
print("Numerical features:  ", numerical_cols)


### 3. Stratified Train / Test Split (80 / 20)


In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_congestion, test_size=0.20, random_state=42, stratify=y_congestion
)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X, y_load, test_size=0.20, random_state=42, stratify=y_load
)

print(f"Congestion Train Set: {X_train_c.shape}, Test Set: {X_test_c.shape}")
print(f"Grid Load Train Set:  {X_train_l.shape}, Test Set: {X_test_l.shape}")


### 4. Build Preprocessing Transformer


In [ ]:
def get_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
            ("num", StandardScaler(), numerical_cols)
        ]
    )


### 5. Train and Evaluate Congestion Models


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
}

trained_pipes = {}
results = []
cms = {}
labels_order = ["Low", "Medium", "High"]

for name, clf in models.items():
    pipe = Pipeline([("prep", get_preprocessor()), ("clf", clf)])
    pipe.fit(X_train_c, y_train_c)
    y_pred = pipe.predict(X_test_c)
    
    acc = accuracy_score(y_test_c, y_pred)
    prec = precision_score(y_test_c, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test_c, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test_c, y_pred, average="weighted", zero_division=0)
    
    trained_pipes[name] = pipe
    cms[name] = confusion_matrix(y_test_c, y_pred, labels=labels_order)
    
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4)
    })

comparison_df = pd.DataFrame(results)
print("=== CONGESTION MODEL COMPARISON ===")
comparison_df


### 6. Confusion Matrices for Congestion Classifiers


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, (name, cm) in enumerate(cms.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=labels_order, yticklabels=labels_order, ax=axes[idx])
    axes[idx].set_title(f"{name}\nConfusion Matrix", fontweight="bold")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

plt.tight_layout()
plt.show()


### 7. Accuracy & F1-Score Comparative Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=comparison_df, x="Model", y="Accuracy", palette="viridis", ax=axes[0])
axes[0].set_title("Model Accuracy Comparison", fontweight="bold")
axes[0].set_ylim(0.8, 1.02)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')

sns.barplot(data=comparison_df, x="Model", y="F1 Score", palette="magma", ax=axes[1])
axes[1].set_title("Model F1-Score Comparison", fontweight="bold")
axes[1].set_ylim(0.8, 1.02)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()


### 8. Train and Evaluate Grid Load Level Model


In [ ]:
load_pipe = Pipeline([
    ("prep", get_preprocessor()),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])
load_pipe.fit(X_train_l, y_train_l)
y_pred_l = load_pipe.predict(X_test_l)

print("=== GRID LOAD MODEL EVALUATION ===")
print("Accuracy: ", round(accuracy_score(y_test_l, y_pred_l), 4))
print("Precision:", round(precision_score(y_test_l, y_pred_l, average='weighted'), 4))
print("Recall:   ", round(recall_score(y_test_l, y_pred_l, average='weighted'), 4))
print("F1 Score: ", round(f1_score(y_test_l, y_pred_l, average='weighted'), 4))
print("\nClassification Report:")
print(classification_report(y_test_l, y_pred_l, labels=labels_order))


### 9. Serialize Trained Models via Joblib


In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(trained_pipes["Logistic Regression"], models_dir / "congestion_logistic.pkl")
joblib.dump(trained_pipes["Decision Tree"], models_dir / "congestion_tree.pkl")
joblib.dump(trained_pipes["Random Forest"], models_dir / "congestion_forest.pkl")
joblib.dump(load_pipe, models_dir / "grid_load_model.pkl")

print(f"Successfully saved all models to: {models_dir.resolve()}")
